## Terra Nova Bay Mooring Comparison - 2017 

### Goal: Use various analytical techniques to compare moorings DITN, D, DITS, DITD, and LDEO in 2017 to refine our understanding of winter-time circulation in mid-depths and abyssal. And water mass transformations that may help us infer circulation patterns 

### Techniques: 

Time series visualization, sliding window lag correlations, wavelet analysis, quiver plots, etc... 

### Background information: 





In [ ]:
# import packages 
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr 
import gsw

In [ ]:
%matplotlib widget

In [ ]:
def Calculate_TEOS10_vars(ds):

    if 'sal' not in ds:     # this skips D/2017/ctd which is not a dataset 
        return ds
    
    ds = ds.copy()  # important: DatasetView → writable Dataset

    ds['abs_sal'] = (
        'time',
        gsw.SA_from_SP(
            SP=ds['sal'].values,
            p=ds['z'].values,
            lon=ds['lon'].values,
            lat=ds['lat'].values,
        )
    )

    ds['theta'] = (
        'time',
        gsw.pt_from_t(
            SA=ds['abs_sal'].values,
            t=ds['temp'].values,
            p=ds['z'].values,
            p_ref=0,
        )
    )

    ds['CT'] = (
        'time',
        gsw.CT_from_pt(
            SA=ds['abs_sal'].values,
            pt=ds['theta'].values,
        )
    )

    ds['sigma0'] = (
        'time',
        gsw.sigma0(
            SA=ds['abs_sal'].values,
            CT=ds['CT'].values,
        )
    )

    return ds


In [ ]:
def TS_template(ax1):
    """ 
    """
    # Create figure
    # fig = plt.figure(figsize=(10, 6),constrained_layout=True)
    # # Top left square
    # ax1 = plt.subplot(1,1,1)

    mint=-3
    maxt=2
    mins=32
    maxs=37

    tempL=np.linspace(mint,maxt)
    salL=np.linspace(mins,maxs)
    Tg, Sg = np.meshgrid(tempL,salL)

    lon = 165
    lat = -75.0112
    p = 1093
    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    # rho = gsw.rho(SA, Tg, 0)
    # sigma_theta = rho - 1000

    # CALCULATE DENSITY (METHOD 2)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing1100 = gsw.t_freezing(gsw.SA_from_SP(salL, 0, lon, lat), p=0, saturation_fraction=0)
    plt.plot(salL, t_freezing1100, 'k--', label='Freezing Line at 0 dbar')

    cs = ax1.contour(Sg, Tg, sigma_theta, colors='k', zorder=1, alpha=0.7, levels=15)
    cl=plt.clabel(cs,fontsize=10,inline=False)

In [ ]:
def calculate_doy(ds):

    if "time" not in ds:
        return ds  # skip nodes without time

    x = ds.time.dt.dayofyear + ds.time.dt.hour / 24
    return ds.assign_coords(time_doy=x)

# ctd_2017 = ctd_2017.map_over_datasets(calculate_doy)
# ctd_2018 = ctd_2018.map_over_datasets(calculate_doy)

In [ ]:
# update matplotlib font sizes so figures are readable 
plt.rcParams.update({
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

In [ ]:
# Load in datatree (DELL)
# moorings = xr.open_datatree('/Users/mblan/Documents/research/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

# Load in datatree (MAC)
moorings = xr.open_datatree('/Users/mackenzie/Documents/Research/PhD/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

In [ ]:
moorings

In [ ]:
# # time is messed up on DITS, I will make my own time axis for now and will confirm this issue with Liv 
# # start: 12/2/2017, end: 8/03/2018, dt = 600s
# start = np.datetime64('2017-02-12T12')
# end   = np.datetime64('2018-03-08T14:00:00')
# dt    = np.timedelta64(600, 's')  # 600 seconds = 10 minutes

# time = np.arange(start, end + dt, dt)

In [ ]:
# time

In [ ]:
# DITS_2017 = moorings['DITS/2017/ctd/bot'].to_dataset().dropna(dim='time')

In [ ]:
# DITS_2017['time'] = time

In [ ]:
# moorings['DITS/2017/ctd/bot'] = DITS_2017

In [ ]:
# drop nan from DITD bottom datasets 2018 - was causing plotting issues before 
# not sure why there were so many nans 
# sampling frequency in dataset was 1 min but should be 2 min based on Liv's paper 
# moorings['DITD/2018/ctd/bot'] = moorings['DITD/2018/ctd/bot'].to_dataset().dropna(dim='time')

# same issue with DITN velocity 2018 
# moorings['DITN/2018/vel/bot'] = moorings['DITN/2018/vel/bot'].to_dataset().dropna(dim='time')

### Data for 2017 and 2018

- All bottom sensor data for moorings (P2, P3, D, DITD, DITN, DITS, TNBD, LDEO)

- Notes: 
    - for 2018 using top sensor on DITD (1122 meters), using bottom sensor for 2017 (1225 meters)
        - bottom sensor exists for 2018 but velocity data cuts out in June and waiting to hear back from Liv on CTD data 
        - DITD usually has just one sensor at bottom 

    - Not using velocity data from P2 / P3 right now because paired with top sensor, not bottom 

In [ ]:
# make a datatree for 2017 
# note: important to make this with copy - if not, moorings datatree will get effected 
ctd_2017= xr.DataTree(name='ctd_2017')
ctd_2017.children = {
    'DITD': moorings['DITD/2017/ctd/bot'].copy(),
    'D': moorings['D/2017/ctd/bot'].copy(),
    'LDEO': moorings['LDEO/2017/ctd/bot'].copy(),
    'DITN': moorings['DITN/2017/ctd/bot'].copy(),
    'DITS': moorings['DITS/2017/ctd/bot'].copy()
}

In [ ]:
# make datatree for 2018 ctd data 
ctd_2018= xr.DataTree(name='ctd_2018')
ctd_2018.children = {
    'DITD': moorings['DITD/2018/ctd/bot'].copy(),
    'TNBD': moorings['TNBD/2018/ctd/bot'].copy(),
    'P2': moorings['P2/2018/ctd/bot'].copy(),
    'P3': moorings['P3/2018/ctd/bot'].copy(),
    'DITN': moorings['DITN/2018/ctd/bot'].copy(),
    'DITS': moorings['DITS/2018/ctd/bot'].copy().sel(time=slice('2018-03-15','2019-01-04'))
}

In [ ]:
# make datatree for 2017 velocity 
vel_2017= xr.DataTree(name='vel_2017')
vel_2017.children = {
    'DITD': moorings['DITD/2017/vel/bot'].copy(),
    'D': moorings['D/2017/vel/bot'].copy().sel(time=slice('2017-02','2018-03-15')),
    'LDEO': moorings['LDEO/2017/vel/bot'].copy(),
    'DITN': moorings['DITN/2017/vel/bot'].copy(),
    'DITS': moorings['DITS/2017/vel/bot'].copy()
}

In [ ]:
# make datatree for 2018 velocity 
# NOTE: Leaving P2/P3 out for now, velocity data is paired with top sensor. No velocity near bottom 
vel_2018= xr.DataTree(name='vel_2018')
vel_2018.children = {
    'DITD': moorings['DITD/2018/vel/bot'].copy(),
    'TNBD': moorings['TNBD/2018/vel/bot'].copy(),
    'DITN': moorings['DITN/2018/vel/bot'].copy(),
    'DITS': moorings['DITS/2018/vel/bot'].copy().sel(time=slice('2018-03','2019-01-04'))
}

In [ ]:
vel_2017 = vel_2017.map_over_datasets(calculate_doy)
vel_2018 = vel_2018.map_over_datasets(calculate_doy)

In [ ]:
# 30 minute rolling average on CTD data 
fig = plt.figure(figsize=(12,10), constrained_layout=True)

def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    plt.xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])
    # plt.xlabel("Month")

plt.suptitle('Mooring DITD 2017 and 2018')

ds1 = ctd_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds2 = ctd_2018['DITD'].to_dataset().sel(time=slice('2018')).rolling(time=10).mean()
plt.subplot(5,1,1)
plt.plot(ds1.time_doy,ds1.sal,label='2017')
plt.plot(ds2.time_doy,ds2.sal,label='2018')
plt.ylabel('Practical \nSalinity [PSU]')
plt.legend()
xaxis_months()

plt.subplot(5,1,2)
plt.plot(ds1.time_doy,ds1.temp,label='2017')
plt.plot(ds2.time_doy,ds2.temp,label='2018')
plt.ylabel("In-Situ \nTemperature [$^\circ$C]")
xaxis_months()

plt.subplot(5,1,3)
plt.plot(ds1.time_doy,ds1.sigma0,label='2017')
plt.plot(ds2.time_doy,ds2.sigma0,label='2018')
plt.ylabel("Potential \nDensity [kg/m$^3$]")
xaxis_months()

plt.subplot(5,1,4)
ds1 = vel_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=2).mean()
ds2 = vel_2018['DITD'].to_dataset().sel(time=slice('2018')).rolling(time=3).mean()
plt.plot(ds1.time_doy,ds1.speed,label='2017')
plt.plot(ds2.time_doy,ds2.speed,label='2018')
plt.ylabel('Speed [cm/s]')
xaxis_months()

plt.subplot(5,1,5)
ds1 = vel_2017['DITD'].to_dataset().sel(time=slice('2017'))
ds2 = vel_2018['DITD'].to_dataset().sel(time=slice('2018'))
q = plt.quiver(
    ds1.time_doy.resample(time='ME').mean().values,np.zeros(len(ds1.time_doy.resample(time='ME').mean())),
    ds1.u.resample(time='ME').mean(),
    ds1.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:blue'
)
q = plt.quiver(
    ds2.time_doy.resample(time='ME').mean().values,np.zeros(len(ds2.time_doy.resample(time='ME').mean())),
    ds2.u.resample(time='ME').mean(),
    ds2.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:orange'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')
plt.ylabel('Speed [m/s]')
# plt.xlabel('Day of Year')
xaxis_months()


# month_starts = [32,60,91,121,152,182,213,244,274,305,335]

# plt.xticks(month_starts,
#            ["Feb","Mar","Apr","May","Jun",
#             "Jul","Aug","Sep","Oct","Nov","Dec"])
# plt.xlabel("Month")

In [ ]:
# 30 min average of CTD data 
fig = plt.figure(figsize=(12,10), constrained_layout=True)
def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    plt.xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])
plt.suptitle('Mooring DITS 2017 and 2018')

ds1 = ctd_2017['DITS'].to_dataset().sel(time=slice('2017')).rolling(time=3).mean()
ds2 = ctd_2018['DITS'].to_dataset().sel(time=slice('2018')).rolling(time=15).mean()
plt.subplot(5,1,1)
plt.plot(ds1.time_doy,ds1.sal,label='2017')
plt.plot(ds2.time_doy,ds2.sal,label='2018')
plt.ylabel('Practical \nSalinity [PSU]')
plt.legend()
xaxis_months()

plt.subplot(5,1,2)
plt.plot(ds1.time_doy,ds1.temp,label='2017')
plt.plot(ds2.time_doy,ds2.temp,label='2018')
plt.ylabel("In-Situ \nTemperature [$^\circ$C]")
xaxis_months()

plt.subplot(5,1,3)
plt.plot(ds1.time_doy,ds1.sigma0,label='2017')
plt.plot(ds2.time_doy,ds2.sigma0,label='2018')
plt.ylabel("Potential \nDensity [kg/m$^3$]")
xaxis_months()

plt.subplot(5,1,4)
ds1 = vel_2017['DITS'].to_dataset().sel(time=slice('2017')).rolling(time=2).mean()
ds2 = vel_2018['DITS'].to_dataset().sel(time=slice('2018')).rolling(time=3).mean()
plt.plot(ds1.time_doy,ds1.speed,label='2017')
plt.plot(ds2.time_doy,ds2.speed,label='2018')
plt.ylabel('Speed [cm/s]')
xaxis_months()

plt.subplot(5,1,5)
ds1 = vel_2017['DITS'].to_dataset().sel(time=slice('2017'))
ds2 = vel_2018['DITS'].to_dataset().sel(time=slice('2018'))
q = plt.quiver(
    ds1.time_doy.resample(time='ME').mean().values,np.zeros(len(ds1.time_doy.resample(time='ME').mean())),
    ds1.u.resample(time='ME').mean(),
    ds1.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:blue'
)
q = plt.quiver(
    ds2.time_doy.resample(time='ME').mean().values,np.zeros(len(ds2.time_doy.resample(time='ME').mean())),
    ds2.u.resample(time='ME').mean(),
    ds2.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:orange'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')
plt.ylabel('Speed [m/s]')
# plt.xlabel('Day of Year')
xaxis_months()


# month_starts = [1,32,60,91,121,152,182,213,244,274,305,335]

# plt.xticks(month_starts,
#            ["Jan","Feb","Mar","Apr","May","Jun",
#             "Jul","Aug","Sep","Oct","Nov","Dec"])
# plt.xlabel("Month")

In [ ]:
# 30 min rolling average on CTD data 
fig = plt.figure(figsize=(12,10), constrained_layout=True)
def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    plt.xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])
plt.suptitle('Mooring DITN 2017 and 2018')

ds1 = ctd_2017['DITN'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds2 = ctd_2018['DITN'].to_dataset().sel(time=slice('2018')).rolling(time=6).mean()
plt.subplot(5,1,1)
plt.plot(ds1.time_doy,ds1.sal,label='2017')
plt.plot(ds2.time_doy,ds2.sal,label='2018')
plt.ylabel('Practical \nSalinity [PSU]')
plt.legend()
xaxis_months()

plt.subplot(5,1,2)
plt.plot(ds1.time_doy,ds1.temp,label='2017')
plt.plot(ds2.time_doy,ds2.temp,label='2018')
plt.ylabel("In-Situ \nTemperature [$^\circ$C]")
xaxis_months()

plt.subplot(5,1,3)
plt.plot(ds1.time_doy,ds1.sigma0,label='2017')
plt.plot(ds2.time_doy,ds2.sigma0,label='2018')
plt.ylabel("Potential \nDensity [kg/m$^3$]")
xaxis_months()

plt.subplot(5,1,4)
ds1 = vel_2017['DITN'].to_dataset().sel(time=slice('2017')).rolling(time=2).mean()
ds2 = vel_2018['DITN'].to_dataset().sel(time=slice('2018')).rolling(time=3).mean()
plt.plot(ds1.time_doy,ds1.speed,label='2017')
plt.plot(ds2.time_doy,ds2.speed,label='2018')
plt.ylabel('Speed [cm/s]')
xaxis_months()

plt.subplot(5,1,5)
ds1 = vel_2017['DITN'].to_dataset().sel(time=slice('2017'))
ds2 = vel_2018['DITN'].to_dataset().sel(time=slice('2018'))
q = plt.quiver(
    ds1.time_doy.resample(time='ME').mean().values,np.zeros(len(ds1.time_doy.resample(time='ME').mean())),
    ds1.u.resample(time='ME').mean(),
    ds1.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:blue'
)
q = plt.quiver(
    ds2.time_doy.resample(time='ME').mean().values,np.zeros(len(ds2.time_doy.resample(time='ME').mean())),
    ds2.u.resample(time='ME').mean(),
    ds2.v.resample(time='ME').mean(),
    scale=70, units='width',color='tab:orange'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')
plt.ylabel('Speed [m/s]')
# plt.xlabel('Day of Year')
xaxis_months()


# month_starts = [1,32,60,91,121,152,182,213,244,274,305,335]

# plt.xticks(month_starts,
#            ["Jan","Feb","Mar","Apr","May","Jun",
#             "Jul","Aug","Sep","Oct","Nov","Dec"])
# plt.xlabel("Month")

In [ ]:
# Now make plots of temperature vs. salinity for each year and zoom into in-phase and out-of-phase sections 
fig = plt.figure(figsize=(12,6), constrained_layout=True)

def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    ax.set_xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])

ds1 = ctd_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds2 = ctd_2018['DITD'].to_dataset().sel(time=slice('2018')).rolling(time=10).mean()

ax = plt.subplot(2,1,1)
# Left axis (temperature)
l1, = ax.plot(ds1.time_doy, ds1.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
xaxis_months()

# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds1.time_doy, ds1.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')

ax.set_title('Mooring DITD 2017 1225m')
# --- Density anomaly (second right axis)
# ax3 = ax1.twinx()

# # move third axis outward so it doesn't overlap
# ax3.spines["right"].set_position(("outward", 60))

# ax3.plot(sigma0_anom.time_doy, sigma0_anom, color='k')
# ax3.set_ylabel('Potential density anomaly (kg m$^{-3}$)', color='k')
# ax3.tick_params(axis='y', labelcolor='k')

ax = plt.subplot(2,1,2)
# Left axis (temperature)
l1, = ax.plot(ds2.time_doy, ds2.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds2.time_doy, ds2.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')
# ax.set_xlabel('Day of Year')
xaxis_months()

ax.set_title('Mooring DITD 2018 1218m')

In [ ]:
fig = plt.figure(figsize=(12,4), constrained_layout=True)

def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    ax.set_xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])

# ds1 = ctd_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds1 = ctd_2017['D'].to_dataset().sel(time=slice('2017'))

ax = plt.subplot(1,1,1)
# Left axis (temperature)
l1, = ax.plot(ds1.time_doy, ds1.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
xaxis_months()

# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds1.time_doy, ds1.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')

ax.set_title('Mooring D 2017 1086m')

In [ ]:
# Now make plots of temperature vs. salinity for each year and zoom into in-phase and out-of-phase sections 
fig = plt.figure(figsize=(12,6), constrained_layout=True)

def xaxis_months():
    month_starts = [32,60,91,121,152,182,213,244,274,305,335]

    ax.set_xticks(month_starts,
            ["Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"])

# ds1 = ctd_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds1 = ctd_2017['DITD'].to_dataset().sel(time=slice('2017')).rolling(time=15).mean()
ds2 = vel_2017['DITD'].to_dataset().sel(time=slice('2017'))

ax = plt.subplot(2,1,1)
# Left axis (temperature)
l1, = ax.plot(ds1.time_doy, ds1.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
xaxis_months()

# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds1.time_doy, ds1.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')

ax.set_title('Mooring DITD 2017 1225m')

ax = plt.subplot(2,1,2)
# Left axis (temperature)
# l1, = ax.plot(ds2.time_doy, ds2.temp, color='tab:red', label='Temperature')
# ax.set_ylabel('Temperature (°C)', color='tab:red')
# ax.tick_params(axis='y', labelcolor='tab:red')
# # Right axis (salinity)
# ax1 = ax.twinx()
# l2, = ax1.plot(ds2.time_doy, ds2.sal, color='tab:blue', label='Salinity')
# ax1.set_ylabel('Salinity (psu)', color='tab:blue')
# ax1.tick_params(axis='y', labelcolor='tab:blue')
# # Combine legends
# lines = [l1, l2]
# labels = [l.get_label() for l in lines]
# ax.legend(lines, labels, loc='lower left')
# ax.set_xlabel('Day of Year')
q = plt.quiver(
    ds2.time_doy.resample(time='W').mean().values,np.zeros(len(ds2.time_doy.resample(time='W').mean())),
    ds2.u.resample(time='W').mean(),
    ds2.v.resample(time='W').mean(),
    scale=150, units='width',color='k'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')
plt.ylabel('Speed [m/s]')
plt.yticks([])

xaxis_months()

ax.set_title('Mooring DITD 2018 1218m')

In [ ]:
TNBD_resampled_ctd = ctd_2018['TNBD'].to_dataset().resample(time='15min').mean()

In [ ]:
TNBD_resampled_vel = vel_2018['TNBD'].to_dataset().resample(time='15min').mean()

In [ ]:
# make TS diagram colored in by velocity for TNBD 
fig = plt.figure(figsize=(10,6), constrained_layout=True)
ax = plt.subplot(1,1,1)

ctd = TNBD_resampled_ctd
vel = TNBD_resampled_vel


# ----- Density Grid -----
mint = ctd.temp.min()
maxt = ctd.temp.max()
mins = ctd.sal.min()
maxs = ctd.sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)



In [ ]:
moorings['DITN/2017/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITN/2017/ctd/bot'].to_dataset())
moorings['DITN/2017/ctd/mid'] = Calculate_TEOS10_vars(moorings['DITN/2017/ctd/mid'].to_dataset())
moorings['DITD/2017/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITD/2017/ctd/bot'].to_dataset())
moorings['D/2017/ctd/bot'] = Calculate_TEOS10_vars(moorings['D/2017/ctd/bot'].to_dataset())
moorings['D/2017/ctd/mid'] = Calculate_TEOS10_vars(moorings['D/2017/ctd/mid'].to_dataset())
moorings['D/2017/ctd/top'] = Calculate_TEOS10_vars(moorings['D/2017/ctd/top'].to_dataset())
# moorings['DITS/2017/ctd/mid'] = Calculate_TEOS10_vars(moorings['DITS/2017/ctd/mid'].to_dataset())   # mid sensor had messed up S data 
moorings['DITS/2017/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITS/2017/ctd/bot'].to_dataset())

In [ ]:
moorings['DITN/2017/ctd/mid'].z.values

In [ ]:
def TS_template(ax1,levels=5):
    """ 
    """
    # Create figure
    # fig = plt.figure(figsize=(10, 6),constrained_layout=True)
    # # Top left square
    # ax1 = plt.subplot(1,1,1)

    mint=-2.5
    maxt=-1
    mins=34
    maxs=35

    tempL=np.linspace(mint,maxt)
    salL=np.linspace(mins,maxs)
    Tg, Sg = np.meshgrid(tempL,salL)

    lon = 165
    lat = -75.0112
    p = 1093
    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    # rho = gsw.rho(SA, Tg, 0)
    # sigma_theta = rho - 1000

    # CALCULATE DENSITY (METHOD 2)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing1100 = gsw.t_freezing(gsw.SA_from_SP(salL, 0, lon, lat), p=0, saturation_fraction=0)
    plt.plot(salL, t_freezing1100, 'k--', label='Freezing Line at 0 dbar')
    
    cs = ax1.contour(Sg, Tg, sigma_theta, colors='k', zorder=10, levels=levels)
    cl=plt.clabel(cs,fontsize=10,inline=True,inline_spacing=0.1)
    

In [ ]:
moorings['DITD/2017/ctd/bot'].z.values

In [ ]:
### publication ready T/S Diagrams 
# WATER MASS DEFINITIONS: TISW (T<-1.94, sigma < 28), HSSW (sal > 34.7, sigma > 28),)
fig = plt.figure(figsize=(10,6), constrained_layout=True)
ax1 = plt.subplot(1,1,1)
TS_template(ax1,levels=10)

# plot LDEO mooring data 
plt.scatter(moorings['LDEO/2017/ctd/bot'].sal,moorings['LDEO/2017/ctd/bot'].theta,s=0.5,color='tab:red',alpha=0.7,label='LDEO')
# DITN
plt.scatter(moorings['DITN/2017/ctd/bot'].sal,moorings['DITN/2017/ctd/bot'].theta,s=0.5,color='tab:green',alpha=0.7,label='DITN')
plt.scatter(moorings['DITN/2017/ctd/mid'].sal,moorings['DITN/2017/ctd/mid'].theta,s=0.5,color='tab:green',alpha=0.7)
# DITD
plt.scatter(moorings['DITD/2017/ctd/bot'].sal,moorings['DITD/2017/ctd/bot'].theta,s=0.5,color='tab:orange',alpha=0.7,label='DITD')
# D 
plt.scatter(moorings['D/2017/ctd/bot'].sal,moorings['D/2017/ctd/bot'].theta,s=0.5,color='tab:purple',alpha=0.7,label='D')
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2017')),s=0.5,color='tab:purple',alpha=0.7)
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2017')),s=0.5,color='tab:purple',alpha=0.7)
# DITS
plt.scatter(moorings['DITS/2017/ctd/bot'].sal,moorings['DITS/2017/ctd/bot'].theta,s=0.5,color='tab:blue',alpha=0.7,label='DITS')
# plt.scatter(moorings['DITS/2017/ctd/mid'].sal,moorings['DITS/2017/ctd/mid'].theta,s=0.5)

plt.xlim(34.4,35)
plt.ylim(-2.3,-1.4)
plt.title('Terra Nova Bay Water Masses Between 275 and 1225 meters, 2017')
plt.xlabel('Practical Salinity [PSU]')
plt.ylabel('Potential Temperature [$^{\circ}$C]')
leg = plt.legend(markerscale=4)
leg.set_zorder(11)
plt.text(34.7,-2,"TISW",fontweight='bold')
plt.text(34.8,-1.95,"HSSW",fontweight='bold')
plt.text(34.58,-1.8,"AASW",fontweight='bold')

In [ ]:
### publication ready T/S Diagrams 
# WATER MASS DEFINITIONS: TISW (T<-1.94, sigma < 28), HSSW (sal > 34.7, sigma > 28),)
fig = plt.figure(figsize=(16,4), constrained_layout=True)
datasets = [moorings['LDEO/2017/ctd/bot'],
            moorings['DITN/2017/ctd/bot'],
            moorings['DITN/2017/ctd/mid'],
            moorings['DITD/2017/ctd/bot'],
            moorings['D/2017/ctd/bot'],
            moorings['D/2017/ctd/mid'].sel(time=slice('2017')),
            moorings['D/2017/ctd/top'].sel(time=slice('2017')),
            moorings['DITS/2017/ctd/bot']
            ]

def plot_params(title='Winter'):
    plt.xlim(34.4,35)
    plt.ylim(-2.3,-1.4)
    plt.title(title)
    # plt.xlabel('Practical Salinity [PSU]')
    plt.text(34.7,-2,"TISW",fontweight='bold')
    plt.text(34.8,-1.95,"HSSW",fontweight='bold')
    plt.text(34.58,-1.8,"AASW",fontweight='bold')

ax1 = plt.subplot(1,4,1)
TS_template(ax1)

for i in range(len(datasets)):
    ds = datasets[i].to_dataset()
    ds = ds.sel(time=ds.time.dt.month.isin([12,1,2]))
    plt.scatter(ds.sal,ds.theta,s=0.5,color='grey',alpha=0.7)
plot_params(title='Summer (DJF)')
leg = plt.legend()
leg.set_zorder(11)
plt.ylabel('Potential Temperature [$^{\circ}$C]')

ax2 = plt.subplot(1,4,2)
TS_template(ax2)

for i in range(len(datasets)):
    ds = datasets[i].to_dataset()
    ds = ds.sel(time=ds.time.dt.month.isin([3,4,5]))
    plt.scatter(ds.sal,ds.theta,s=0.5,color='grey',alpha=0.7)
plot_params(title='Fall (MAM)')

ax3 = plt.subplot(1,4,3)
TS_template(ax3)

for i in range(len(datasets)):
    ds = datasets[i].to_dataset()
    ds = ds.sel(time=ds.time.dt.month.isin([6,7,8]))
    plt.scatter(ds.sal,ds.theta,s=0.5,color='grey',alpha=0.7)
plot_params(title='Winter (JJA)')

ax4 = plt.subplot(1,4,4)
TS_template(ax4)

for i in range(len(datasets)):
    ds = datasets[i].to_dataset()
    ds = ds.sel(time=ds.time.dt.month.isin([9,10,11]))
    plt.scatter(ds.sal,ds.theta,s=0.5,color='grey',alpha=0.7)
plot_params(title='Spring (SON)')

fig.supxlabel('Practical Salinity [PSU]')
# plt.savefig('TS_2017_Seasonal.png')

In [ ]:
moorings['DITN/2018/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITN/2018/ctd/bot'].to_dataset())
moorings['DITN/2018/ctd/mid'] = Calculate_TEOS10_vars(moorings['DITN/2018/ctd/mid'].to_dataset())
moorings['DITD/2018/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITD/2018/ctd/bot'].to_dataset())
moorings['DITD/2018/ctd/top'] = Calculate_TEOS10_vars(moorings['DITD/2018/ctd/top'].to_dataset())
# moorings['D/2017/ctd/bot'] = Calculate_TEOS10_vars(moorings['D/2017/ctd/bot'].to_dataset())
# moorings['D/2018/ctd/mid'] = Calculate_TEOS10_vars(moorings['D/2018/ctd/mid'].to_dataset())
# moorings['D/2018/ctd/top'] = Calculate_TEOS10_vars(moorings['D/2018/ctd/top'].to_dataset())
# moorings['DITS/2017/ctd/mid'] = Calculate_TEOS10_vars(moorings['DITS/2017/ctd/mid'].to_dataset())   # mid sensor had messed up S data 
moorings['DITS/2018/ctd/bot'] = Calculate_TEOS10_vars(moorings['DITS/2018/ctd/bot'].to_dataset())

In [ ]:
### publication ready T/S Diagrams 
# WATER MASS DEFINITIONS: TISW (T<-1.94, sigma < 28), HSSW (sal > 34.7, sigma > 28),)
fig = plt.figure(figsize=(10,6), constrained_layout=True)
ax1 = plt.subplot(1,1,1)
TS_template(ax1,levels=10)
# P2 P3 
# plt.scatter(moorings['P2/2018/ctd/bot'].sal.sel(time=slice('2018')),moorings['P2/2018/ctd/bot'].theta.sel(time=slice('2018')),s=0.5,color='tab:pink',alpha=0.7,label='P2')
# plt.scatter(moorings['P3/2018/ctd/bot'].sal.sel(time=slice('2018')),moorings['P3/2018/ctd/bot'].theta.sel(time=slice('2018')),s=0.5,color='tab:olive',alpha=0.7,label='P3')

# 2017 DATA 
plt.scatter(moorings['LDEO/2017/ctd/bot'].sal,moorings['LDEO/2017/ctd/bot'].theta,s=0.5,color='k',alpha=0.4,label='LDEO')
# DITN
plt.scatter(moorings['DITN/2017/ctd/bot'].sal,moorings['DITN/2017/ctd/bot'].theta,s=0.5,color='k',alpha=0.4,label='DITN')
plt.scatter(moorings['DITN/2017/ctd/mid'].sal,moorings['DITN/2017/ctd/mid'].theta,s=0.5,color='k',alpha=0.4)
# DITD
plt.scatter(moorings['DITD/2017/ctd/bot'].sal,moorings['DITD/2017/ctd/bot'].theta,s=0.5,color='k',alpha=0.4,label='DITD')
# D 
plt.scatter(moorings['D/2017/ctd/bot'].sal,moorings['D/2017/ctd/bot'].theta,s=0.5,color='k',alpha=0.4,label='D')
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2017')),s=0.5,color='k',alpha=0.4)
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2017')),s=0.5,color='k',alpha=0.4)
# DITS
plt.scatter(moorings['DITS/2017/ctd/bot'].sal,moorings['DITS/2017/ctd/bot'].theta,s=0.5,color='k',alpha=0.4,label='DITS')

# 2018 DATA
plt.scatter(moorings['TNBD/2018/ctd/bot'].sal,moorings['TNBD/2018/ctd/bot'].theta,s=0.5,color='grey',alpha=0.7,label='TNBD')
plt.scatter(moorings['TNBD/2018/ctd/top'].sal,moorings['TNBD/2018/ctd/top'].theta,s=0.5,color='grey',alpha=0.7)
# DITN
plt.scatter(moorings['DITN/2018/ctd/bot'].sal,moorings['DITN/2018/ctd/bot'].theta,s=0.5,color='grey',alpha=0.7,label='DITN')
plt.scatter(moorings['DITN/2018/ctd/mid'].sal,moorings['DITN/2018/ctd/mid'].theta,s=0.5,color='grey',alpha=0.7)
# DITD
plt.scatter(moorings['DITD/2018/ctd/bot'].sal,moorings['DITD/2018/ctd/bot'].theta,s=0.5,color='grey',alpha=0.7)
plt.scatter(moorings['DITD/2018/ctd/top'].sal,moorings['DITD/2018/ctd/top'].theta,s=0.5,color='grey',alpha=0.7,label='DITD')
# D 
# plt.scatter(moorings['D/2017/ctd/bot'].sal,moorings['D/2017/ctd/bot'].theta,s=0.5,color='tab:purple',alpha=0.7,label='D')
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2018')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2018')),s=0.5,color='grey',alpha=0.7,label='D')
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2018')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2018')),s=0.5,color='grey',alpha=0.7)
# DITS
plt.scatter(moorings['DITS/2018/ctd/bot'].sal.sel(time=slice('2018')),moorings['DITS/2018/ctd/bot'].theta.sel(time=slice('2018')),s=0.5,color='grey',alpha=0.7,label='DITS')
# plt.scatter(moorings['DITS/2017/ctd/mid'].sal,moorings['DITS/2017/ctd/mid'].theta,s=0.5)

plt.xlim(34.4,35)
plt.ylim(-2.3,-1.4)
plt.title('Terra Nova Bay Water Masses 2017 (black) vs. 2018 (grey)')
plt.xlabel('Practical Salinity [PSU]')
plt.ylabel('Potential Temperature [$^{\circ}$C]')
# leg = plt.legend(markerscale=4)
# leg.set_zorder(11)
plt.text(34.7,-2,"TISW",fontweight='bold',color='tab:red')
plt.text(34.8,-2,"HSSW",fontweight='bold',color='tab:red')
plt.text(34.58,-1.8,"AASW",fontweight='bold',color='tab:red')
plt.text(34.7,-0.5,"CDW / MCDW",fontweight='bold',color='tab:red')

In [ ]:
# TS Diagram Mooring D 

fig = plt.figure(figsize=(6,6), constrained_layout=True)
ax1 = plt.subplot(1,1,1)
TS_template(ax1,levels=10)


# D 2018 
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2018')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2018')),s=0.5,color='tab:orange',label='2018')
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2018')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2018')),s=0.5,color='tab:orange')

# D 2017 
plt.scatter(moorings['D/2017/ctd/bot'].sal,moorings['D/2017/ctd/bot'].theta,s=0.5,color='tab:blue',label='2017')
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2017')),s=0.5,color='tab:blue')
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2017')),s=0.5,color='tab:blue')

plt.xlim(34.4,35)
plt.ylim(-2.3,-1.4)
plt.title('Mooring D 2017 and 2018 550-1088m')
plt.xlabel('Practical Salinity [PSU]')
plt.ylabel('Potential Temperature [$^{\circ}$C]')
leg = plt.legend(markerscale=4)
leg.set_zorder(11)
plt.text(34.7,-2,"TISW",fontweight='bold',color='tab:red')
plt.text(34.8,-1.95,"HSSW",fontweight='bold',color='tab:red')
plt.text(34.58,-1.8,"AASW",fontweight='bold',color='tab:red')
plt.text(34.7,-0.5,"CDW / MCDW",fontweight='bold',color='tab:red')

In [ ]:
moorings['DITN/2018/ctd/bot']

In [ ]:
# TS Diagram Mooring D 

fig = plt.figure(figsize=(6,6), constrained_layout=True)
ax1 = plt.subplot(1,1,1)
TS_template(ax1,levels=10)




# DITN
plt.scatter(moorings['DITN/2018/ctd/bot'].sal,moorings['DITN/2018/ctd/bot'].theta,s=0.5,color='tab:orange',label='2018')
plt.scatter(moorings['DITN/2018/ctd/mid'].sal,moorings['DITN/2018/ctd/mid'].theta,s=0.5,color='tab:orange')

# DITN
plt.scatter(moorings['DITN/2017/ctd/bot'].sal,moorings['DITN/2017/ctd/bot'].theta,s=0.5,color='tab:blue',label='2017')
plt.scatter(moorings['DITN/2017/ctd/mid'].sal,moorings['DITN/2017/ctd/mid'].theta,s=0.5,color='tab:blue')

plt.xlim(34.4,35)
plt.ylim(-2.3,-1.4)
plt.title('Mooring DITN 2017 and 2018, 275-675m')
plt.xlabel('Practical Salinity [PSU]')
plt.ylabel('Potential Temperature [$^{\circ}$C]')
leg = plt.legend(markerscale=4)
leg.set_zorder(11)
plt.text(34.7,-2,"TISW",fontweight='bold',color='tab:red')
plt.text(34.8,-1.95,"HSSW",fontweight='bold',color='tab:red')
plt.text(34.58,-1.8,"AASW",fontweight='bold',color='tab:red')
plt.text(34.7,-0.5,"CDW / MCDW",fontweight='bold',color='tab:red')

In [ ]:
### publication ready T/S Diagrams 
# WATER MASS DEFINITIONS: TISW (T<-1.94, sigma < 28), HSSW (sal > 34.7, sigma > 28),)
fig = plt.figure(figsize=(10,6), constrained_layout=True)
ax1 = plt.subplot(1,1,1)
TS_template(ax1,levels=10)

# plot LDEO mooring data 
plt.scatter(moorings['LDEO/2017/ctd/bot'].sal,moorings['LDEO/2017/ctd/bot'].theta,s=0.5,color='tab:red',alpha=0.7,label='LDEO')
# DITN
plt.scatter(moorings['DITN/2017/ctd/bot'].sal,moorings['DITN/2017/ctd/bot'].theta,s=0.5,color='tab:green',alpha=0.7,label='DITN')
plt.scatter(moorings['DITN/2017/ctd/mid'].sal,moorings['DITN/2017/ctd/mid'].theta,s=0.5,color='tab:green',alpha=0.7)
# DITD
plt.scatter(moorings['DITD/2017/ctd/bot'].sal,moorings['DITD/2017/ctd/bot'].theta,s=0.5,color='tab:orange',alpha=0.7,label='DITD')
# D 
plt.scatter(moorings['D/2017/ctd/bot'].sal,moorings['D/2017/ctd/bot'].theta,s=0.5,color='tab:purple',alpha=0.7,label='D')
plt.scatter(moorings['D/2017/ctd/mid'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/mid'].theta.sel(time=slice('2017')),s=0.5,color='tab:purple',alpha=0.7)
plt.scatter(moorings['D/2017/ctd/top'].sal.sel(time=slice('2017')),moorings['D/2017/ctd/top'].theta.sel(time=slice('2017')),s=0.5,color='tab:purple',alpha=0.7)
# DITS
plt.scatter(moorings['DITS/2017/ctd/bot'].sal,moorings['DITS/2017/ctd/bot'].theta,s=0.5,color='tab:blue',alpha=0.7,label='DITS')
# plt.scatter(moorings['DITS/2017/ctd/mid'].sal,moorings['DITS/2017/ctd/mid'].theta,s=0.5)

plt.xlim(34.4,35)
plt.ylim(-2.3,-1.4)
plt.title('Terra Nova Bay Water Masses Between 275 and 1225 meters, 2017')
plt.xlabel('Practical Salinity [PSU]')
plt.ylabel('Potential Temperature [$^{\circ}$C]')
leg = plt.legend(markerscale=4)
leg.set_zorder(11)
plt.text(34.7,-2,"TISW",fontweight='bold')
plt.text(34.8,-1.95,"HSSW",fontweight='bold')
plt.text(34.58,-1.8,"AASW",fontweight='bold')

In [ ]:
# time series looking at salinity jump 
fig = plt.figure(figsize=(12,4),constrained_layout=True)

ax = plt.subplot(2,1,1)
plt.plot(moorings['DITD/2017/ctd/bot'].time.sel(time=slice('2017')),moorings['DITD/2017/ctd/bot'].sal.sel(time=slice('2017')),label='DITD')
plt.plot(moorings['D/2017/ctd/bot'].time.sel(time=slice('2017')),moorings['D/2017/ctd/bot'].sal.sel(time=slice('2017')),label='D')
plt.plot(moorings['DITS/2017/ctd/bot'].time.sel(time=slice('2017')),moorings['DITS/2017/ctd/bot'].sal.sel(time=slice('2017')),label='DITS')
plt.plot(moorings['DITN/2017/ctd/bot'].time.sel(time=slice('2017')),moorings['DITN/2017/ctd/bot'].sal.sel(time=slice('2017')),label='DITN')
plt.ylim(34.7,34.9)
plt.legend()

plt.subplot(2,1,2)
plt.plot(moorings['DITD/2018/ctd/bot'].time,moorings['DITD/2018/ctd/bot'].sal,label='DITD')
plt.plot(moorings['TNBD/2018/ctd/bot'].time,moorings['TNBD/2018/ctd/bot'].sal,label='TNBD',color='tab:purple')
plt.plot(moorings['DITS/2018/ctd/bot'].time.sel(time=slice('2018')),moorings['DITS/2018/ctd/bot'].sal.sel(time=slice('2018')),label='DITS')
plt.plot(moorings['DITN/2018/ctd/bot'].time,moorings['DITN/2018/ctd/bot'].sal,label='DITN')
plt.ylim(34.7,34.9)
plt.legend()

In [ ]:
# plot Sal signal at each mooring to compare 2017 and 2018 

fig = plt.figure(figsize=(12,6))

ax = plt.subplot(211)
ds = moorings['DITD/2017/ctd/bot'].to_dataset()

plt.plot(ds.time,ds.sal)
ds = moorings['DITD/2018/ctd/bot'].to_dataset()

ax1 = plt.subplot(212)
ax1.plot(ds.time,ds.sal)

plt.suptitle('DITD')

In [ ]:
fig = plt.figure(figsize=(12,6))

ax = plt.subplot(211)
ds = moorings['DITN/2017/ctd/bot'].to_dataset()

plt.plot(ds.time,ds.sal)
ds = moorings['DITN/2018/ctd/bot'].to_dataset()

ax1 = plt.subplot(212)
ax1.plot(ds.time,ds.sal)

plt.suptitle('DITN')

In [ ]:
fig = plt.figure(figsize=(12,6))

ax = plt.subplot(211)
ds = moorings['DITS/2017/ctd/bot'].to_dataset()

plt.plot(ds.time,ds.sal)
ds = moorings['DITS/2018/ctd/bot'].to_dataset().sel(time=slice('2018'))

ax1 = plt.subplot(212)
ax1.plot(ds.time,ds.sal)
ax1.set_ylim(34.8,34.86)
plt.suptitle('DITS')

In [ ]:
ds

In [ ]:
moorings['LDEO/2017/ctd/bot'].lat.values,moorings['LDEO/2017/ctd/bot'].lon.values

In [ ]:
moorings['DITD/2017/ctd/bot'].lat.values,moorings['DITD/2017/ctd/bot'].lon.values

In [ ]:
moorings['DITD/2017/vel/bot'].speed.mean()

In [ ]:
fig = plt.figure(figsize=(12,4))

ax = plt.subplot(111)
ds = moorings['LDEO/2017/ctd/bot'].to_dataset().sel(time=slice('2017-07','2017-11'))
plt.plot(ds.time,ds.sal,label='sal')

ax1 = ax.twinx()
ax1.plot(ds.time,ds.temp,color='tab:red',label='temp')

plt.legend()
ax.axhline(y=34.8,color='k',linestyle='dashed',zorder=1)

In [ ]:
# load in other DITD data to compare with 
DITD_raw = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

In [ ]:
moorings['DITD/2018/ctd/bot'].time

In [ ]:
DITD_raw

In [ ]:
import matplotlib.ticker as mticker


In [ ]:
# looking for a signal at 07/21 ish 
fig = plt.figure(figsize=(10,3),constrained_layout=True)

ax = plt.subplot(111)
ds = DITD_raw.sel(time=slice('2018-07','2018-11'))
plt.plot(ds.time,ds.practical_salinity,label='sal')

ax1 = ax.twinx()
ax1.plot(ds.time,ds.temp,color='tab:red',label='temp')
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.legend()
# ax.axhline(y=34.8,color='k',linestyle='dashed',zorder=1)
plt.title('DITD Data Accessed From Zappa Servers')

In [ ]:
moorings['DITD/2018/vel/top']

In [ ]:
fig = plt.figure(figsize=(12,4))
moorings['TNBD/2018/vel/bot'].u.plot()
moorings['TNBD/2018/vel/top'].u.plot()

In [ ]:
ds

In [ ]:
# looking for a signal at 07/21 ish 
fig = plt.figure(figsize=(12,4))

ax = plt.subplot(111)
ds = moorings['DITD/2018/ctd/top'].to_dataset()
plt.plot(ds.time,ds.pres,label='sal')
# plt.ylim(34.78,34.88)

In [ ]:
# looking for a signal at 07/21 ish 
fig = plt.figure(figsize=(12,4))

ax = plt.subplot(111)
ds = moorings['DITD/2018/ctd/bot'].to_dataset()
plt.plot(ds.time,ds.sal,label='sal')
plt.ylim(34.78,34.88)

In [ ]:
# looking for a signal at 07/21 ish 
fig = plt.figure(figsize=(12,4))

ax = plt.subplot(111)
ds = moorings['DITD/2018/ctd/bot'].to_dataset().sel(time=slice('2018-07','2018-11'))
plt.plot(ds.time,ds.sal,label='sal')

ax1 = ax.twinx()
ax1.plot(ds.time,ds.temp,color='tab:red',label='temp')
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.legend()
# ax.axhline(y=34.8,color='k',linestyle='dashed',zorder=1)
plt.title('DITD Data Accessed From SEANOE')

### Stopped here! 



In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(test.time, test.sal)

In [ ]:
test.groupby("time.month")

In [ ]:
test = ctd_2017['DITD'].to_dataset().assign_coords(month=ctd_2017['DITD'].time.dt.month)

In [ ]:
# load in DITD data 
# ditd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
ditd_vel1200 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1200.nc')

# ditd_sbe1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1100.nc')
ditd_sbe1200 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

In [ ]:
# verify DITD data 
fig = plt.figure(figsize=(12,4))
# plt.plot(ctd_2018['DITD'].time,ctd_2018['DITD'].sal)

# plt.plot(ditd_sbe1200.time,ditd_sbe1200.practical_salinity)

plt.plot(vel_2018['DITD'].time, vel_2018['DITD'].speed)

# This is the messed up data, need to bring in raw data I was previously working with 


### Investigate 2017 

In [ ]:
moorings['D/2017/ctd/bot']

In [ ]:
moorings['D/2017/vel/bot']

In [ ]:
fig = plt.figure(figsize=(12,10))

plt.subplot(3,1,1)
plt.plot(moorings['D/2017/ctd/bot'].time,moorings['D/2017/ctd/bot'].sal,label='D')
plt.plot(moorings['DITD/2017/ctd/bot'].time,moorings['DITD/2017/ctd/bot'].sal,label='DITD')
plt.plot(moorings['DITN/2017/ctd/bot'].time,moorings['DITN/2017/ctd/bot'].sal,label='DITN')
plt.plot(moorings['DITS/2017/ctd/bot'].time,moorings['DITS/2017/ctd/bot'].sal,label='DITS')
plt.legend()

plt.subplot(3,1,2)
plt.plot(moorings['D/2017/ctd/bot'].time,moorings['D/2017/ctd/bot'].temp,label='D')
plt.plot(moorings['DITD/2017/ctd/bot'].time,moorings['DITD/2017/ctd/bot'].temp,label='DITD')
plt.plot(moorings['DITN/2017/ctd/bot'].time,moorings['DITN/2017/ctd/bot'].temp,label='DITN')
plt.plot(moorings['DITS/2017/ctd/bot'].time,moorings['DITS/2017/ctd/bot'].temp,label='DITS')

plt.subplot(3,1,3)
plt.plot(moorings['D/2017/vel/bot'].time.sel(time=slice('2017-03','2018-03')),
         moorings['D/2017/vel/bot'].Mod_0.sel(time=slice('2017-03','2018-03')),label='D')
plt.plot(moorings['DITD/2017/vel/bot'].time,moorings['DITD/2017/vel/bot'].speed,label='DITD')
plt.plot(moorings['DITN/2017/vel/bot'].time,moorings['DITN/2017/vel/bot'].speed,label='DITN')
plt.plot(moorings['DITS/2017/vel/bot'].time,moorings['DITS/2017/vel/bot'].speed,label='DITS')


In [ ]:
# mooring D 
ds = moorings['DITS/2018/ctd/bot']
fig,ax = plt.subplots(figsize=(12,4))

# Left axis (temperature)
l1, = ax.plot(ds.time, ds.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
ax.set_ylim(-1.90,-1.88)

# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds.time, ds.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_ylim(34.7,34.86)

# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')

ax.set_title('Mooring DITD 2017 1200m')

# plt.savefig('MooringDITD_temp_sal.png')

In [ ]:
# mooring D 
ds = moorings['DITD/2017/ctd/bot']
fig,ax = plt.subplots(figsize=(12,4))

# Left axis (temperature)
l1, = ax.plot(ds.time, ds.temp, color='tab:red', label='Temperature')
ax.set_ylabel('Temperature (°C)', color='tab:red')
ax.tick_params(axis='y', labelcolor='tab:red')
ax.set_ylim(-1.90,-1.88)

# Right axis (salinity)
ax1 = ax.twinx()
l2, = ax1.plot(ds.time, ds.sal, color='tab:blue', label='Salinity')
ax1.set_ylabel('Salinity (psu)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_ylim(34.7,34.86)

# Combine legends
lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='lower left')

ax.set_title('Mooring DITD 2017 1200m')



In [ ]:
fig = plt.figure(figsize=(12,12),constrained_layout=True)

plt.subplot(4,1,1)
ds = moorings['DITD/2017/vel/bot']
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITD 1239 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,2)
ds = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2017-03','2018-03'))
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('D 1135 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,3)
ds = moorings['DITN/2017/vel/bot']
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITN 675 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,4)
ds = moorings['DITS/2017/vel/bot']
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITS 1088 m Velocity')
plt.ylabel('Velocity (m/s)')

# plt.savefig('veloctiy_deep_sensors_2017.png')

### Previous Work 

In [ ]:
# mooring selection
# we want: 2017 data for D, DITN, LDEO, and DITS
# D @ 
D_2017 = moorings['D/2017']
DITN_2017 = moorings['DITN/2017']
DITS_2017 = moorings['DITS/2017']
LDEO = moorings['LDEO/2017']
DITD_2017 = moorings['DITD/2017']

In [ ]:
DITD_2017.vel.bot.z.values

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITD_2017.ctd.bot
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITD_2017.vel.bot
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITD 2017 1239 meters')

In [ ]:
DITS_2017.ctd.bot

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITS_2017.ctd.bot
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITS_2017.vel.bot
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITS 2017 1088 meters')

#### Question: 

Why does DITS ctd data stop in 2017-09? 



In [ ]:
ds.z.values

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITS_2017.ctd.top
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITS_2017.vel.top
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITS 2017 191 meters')

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITS_2017.ctd.mid
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITS_2017.vel.mid
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITS 2017 384 meters')

#### Question:

why is there no salinity data at DITS 384 meters in 2017? This data has been cited in Stevens et al. 2024 yet Liv's paper states "faulty P and S data removed" 

In [ ]:
ds.z.values

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITN_2017.ctd.top
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITN_2017.vel.top
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITN 2017 76 meters')

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITN_2017.ctd.mid
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITN_2017.vel.mid
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITN 2017 271 meters')

In [ ]:
ds.z.values

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = DITN_2017.ctd.bot
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = DITN_2017.vel.bot
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITN 2017 694 meters')

In [ ]:
D_2017.ctd.top

In [ ]:
def Calculate_TEOS10_vars(ds):

    if 'sal' not in ds:     # this skips D/2017/ctd which is not a dataset 
        return ds
    
    ds = ds.copy()  # important: DatasetView → writable Dataset

    ds['abs_sal'] = (
        'time',
        gsw.SA_from_SP(
            SP=ds['sal'].values,
            p=ds['z'].values,
            lon=ds['lon'].values,
            lat=ds['lat'].values,
        )
    )

    ds['theta'] = (
        'time',
        gsw.pt_from_t(
            SA=ds['abs_sal'].values,
            t=ds['temp'].values,
            p=ds['z'].values,
            p_ref=0,
        )
    )

    ds['CT'] = (
        'time',
        gsw.CT_from_pt(
            SA=ds['abs_sal'].values,
            pt=ds['theta'].values,
        )
    )

    ds['sigma0'] = (
        'time',
        gsw.sigma0(
            SA=ds['abs_sal'].values,
            CT=ds['CT'].values,
        )
    )

    return ds


In [ ]:
D_2017.ctd = D_2017.ctd.map_over_datasets(Calculate_TEOS10_vars)

In [ ]:
D_2017.ctd

In [ ]:
# # calculate potential temp, absolute sal, conservative temp, etc.. 

# abs_sal = gsw.SA_from_SP(SP=D_2017.ctd.top.sal,
#                p = D_2017.ctd.top.z.values,
#                lon = D_2017.ctd.top.lon.values,
#                lat = D_2017.ctd.top.lat.values)


# theta = gsw.pt_from_t(SA=abs_sal, t=D_2017.ctd.top.temp, p = D_2017.ctd.top.z.values, p_ref=0)

# CT = gsw.CT_from_pt(SA=abs_sal, pt = theta)

# sigma0 = gsw.sigma0(SA=abs_sal, CT=CT)



In [ ]:
D_2017.vel

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

ds = D_2017.ctd.top
plt.subplot(5,1,1)
plt.plot(ds.time,ds.sal,color='tab:green')
plt.ylabel('Practical Salinity \n [PSU]')

plt.subplot(5,1,2)
plt.plot(ds.time,ds.temp, color='tab:red')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(5,1,3)
plt.plot(ds.time,ds.sigma0, color='tab:purple')
plt.ylabel('Potential Density \n [kg/m^3]')

ds = D_2017.vel.top
plt.subplot(5,1,4)
plt.plot(ds.time,ds.speed, color='tab:blue')
plt.ylabel('Current Speed \n [cm/s]')

plt.subplot(5,1,5)
plt.scatter(ds.time,ds.dir,s=0.5, color='tab:orange')
plt.ylabel('Current Direction \n [Degrees]')

plt.suptitle('DITN 2017 76 meters')

In [ ]:
ds.z.values

In [ ]:
LDEO.ctd.bot

In [ ]:
DITN_2017.ctd.bot.z.values

In [ ]:
fig = plt.figure(figsize=(10,10), constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(D_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),D_2017.ctd.bot,label='D')
plt.plot(DITD_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITD_2017.ctd.bot, label='DITD')
plt.plot(DITS_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITS_2017.ctd.bot,label='DITS')
plt.plot(DITN_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')), DITN_2017.ctd.bot,label='DITN')
# plt.plot(LDEO.ctd.bot.time.sel(time=slice('2017-07','2017-11')),LDEO.ctd.bot,label='LDEO')
plt.ylabel('Practical Salinity \n [PSU]')
plt.legend()

plt.subplot(3,1,2)
plt.plot(D_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),D_2017.ctd.bot.temp,label='D')
plt.plot(DITD_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITD_2017.ctd.bot.temp, label='DITD')
plt.plot(DITS_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITS_2017.ctd.bot.temp,label='DITS')
plt.plot(DITN_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')), DITN_2017.ctd.bot.temp,label='DITN')
plt.ylabel('In-situ Temperature \n [Deg C]')

plt.subplot(3,1,3)
plt.plot(D_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),D_2017.ctd.bot.sigma0,label='D 1086m')
plt.plot(DITD_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITD_2017.ctd.bot.sigma0, label='DITD 1225m')
plt.plot(DITS_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')),DITS_2017.ctd.bot.sigma0,label='DITS 1088m')
plt.plot(DITN_2017.ctd.bot.time.sel(time=slice('2017-07','2017-11')), DITN_2017.ctd.bot.sigma0,label='DITN 675m')
plt.ylabel('Potential Density \n [kg/m^3]')
plt.legend()

# plt.savefig('Salinity_deep_sensors_2017.png')

In [ ]:
# time is fucked up on DITS, I will make my own time axis for now and will confirm this issue with Liv 
# start: 12/2/2017, end: 8/03/2018, dt = 600s
start = np.datetime64('2017-02-12T12')
end   = np.datetime64('2018-03-08T14:00:00')
dt    = np.timedelta64(600, 's')  # 600 seconds = 10 minutes

time = np.arange(start, end + dt, dt)

In [ ]:
time

In [ ]:
fig = plt.figure()
DITS_2017.ctd.bot.pres.plot()

In [ ]:
DITS_2017.ctd.bot

In [ ]:
moorings['DITS/2017/ctd/bot'] = moorings['DITS/2017/ctd/bot'].to_dataset().dropna(dim='time')

In [ ]:
moorings['DITS/2017/ctd/bot']['time'] = time

In [ ]:
DITS_2017.ctd.bot['time'] = time

In [ ]:
LDEO.ctd.bot

### Map showing quiver plot. 

In [ ]:
bath = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/gebco_2024_n-66.2651_s-87.7985_w159.418_e213.2403.nc')

In [ ]:
import rioxarray
img = rioxarray.open_rasterio('/Users/mackenzie/Documents/Quantarctica3/MackenzieBlanusa/snapshot-2017-09-14T00_00_00Z.tif')  # shape: (bands, y, x)

In [ ]:
# Normalize to 0–1 range for matplotlib
rgba = img / 255.0  # now shape is (4, y, x)

# Transpose to (y, x, band) for imshow
rgba = rgba.transpose("y", "x", "band")

In [ ]:
import geopandas as gpd

In [ ]:
# load shapefile (this was obtained from Quantarctica detailed basemap --> coastlines)
coast = gpd.read_file('/Users/mackenzie/Downloads/antarctic_coastlines.shp')

# clip shapefile 
# Longitude range: 160°E to 165°E
# Latitude range: -75.3° to -74.2°
# bbox = {
#     "minx": 160,
#     "maxx": 170,
#     "miny": -76,
#     "maxy": -74
# }


bbox = {
    "minx": 155,  # wider Ross Sea
    "maxx": 170,
    "miny": -78,
    "maxy": -72
}

In [ ]:
from shapely.geometry import box

In [ ]:
# Check the CRS
print(coast.crs)  # if it's not EPSG:4326, reproject it
if coast.crs.to_string() != "EPSG:4326":
    coast = coast.to_crs("EPSG:4326")

# Define bounding box as a shapely geometry
roi = box(bbox["minx"], bbox["miny"], bbox["maxx"], bbox["maxy"])

# Clip to Terra Nova Bay region
clipped = coast[coast.intersects(roi)]

In [ ]:
# load moorings and try to plot quiver on map 
ds = moorings['D/2017/ctd/bot']

In [ ]:
fig = plt.figure(figsize=(12,4))
ds.sal.plot()

In [ ]:
vel = moorings['D/2017/vel/bot']

In [ ]:
u = vel.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').u
v = vel.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').v

In [ ]:
u.lon.values

In [ ]:
# define u and v's 
vel_D = moorings['D/2017/vel/bot']
u_D = vel_D.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').u
v_D = vel_D.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').v

vel_DITS = moorings['DITS/2017/vel/bot']
u_DITS = vel_DITS.sel(time=slice('2017-09-01','2017-09-08')).mean(dim='time').u
v_DITS = vel_DITS.sel(time=slice('2017-09-01','2017-09-08')).mean(dim='time').v

vel_DITD = moorings['DITD/2017/vel/bot']
u_DITD = vel_DITD.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').u
v_DITD = vel_DITD.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').v

vel_DITN = moorings['DITN/2017/vel/bot']
u_DITN = vel_DITN.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').u
v_DITN = vel_DITN.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').v

vel_LDEO = moorings['LDEO/2017/vel/bot']
u_LDEO = vel_LDEO.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').u
v_LDEO = vel_LDEO.sel(time=slice('2017-08-28','2017-09-04')).mean(dim='time').v



In [ ]:
u_LDEO.lat.values

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as mpatches

# === Main map ===
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([162, 170, -76, -74])  # Terra Nova Bay

# Define extent from the dataset’s coordinates
extent = [float(img.x.min()), float(img.x.max()),
          float(img.y.min()), float(img.y.max())]

# Plot the image
ax.imshow(rgba, origin='upper', extent=extent, transform=ccrs.PlateCarree(),zorder=1)

# Fill the coastline (land/ice) polygons
clipped.plot(ax=ax, facecolor='white', edgecolor='none', transform=ccrs.PlateCarree(), zorder=1)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue',zorder=0)

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

# Convert elevation to depth (positive down)
depth = -bath['elevation']

# Add contours to Cartopy plot

# levels = 10
levels = [100, 300, 500, 800, 1000, 1200, 1500, 2000, 2500, 3000]
contours = ax.contour(
    bath['lon'], bath['lat'], depth,
    levels=levels,
    colors='black',
    linewidths=0.7,
    transform=ccrs.PlateCarree(),  # very important
    zorder=2
)
# plt.clabel(contours, fmt='%d m', fontsize=8)

# plt.clabel(
#     contours,
#     contours.levels,   # <-- force every contour level to get at least one label
#     fmt='%d m',
#     fontsize=8,
#     inline=True
# )

plt.clabel(contours, fmt='%d m', fontsize=8, inline=True, inline_spacing=1)

# plt.clabel(
#     contours,
#     contours.levels,
#     fmt='%d m',
#     fontsize=8,
#     manual=[(174, -72),(174, -74),(174,-71.8)]  # (lon, lat) points where labels should go
# )

# plt.clabel(contours, fmt='%d m', fontsize=8, inline=True, inline_spacing=1)

# ax.scatter(tnbd_vel1100.lon, tnbd_vel1100.lat, color='red', transform=ccrs.PlateCarree(), label="TNBD",zorder=10)
# ax.scatter(ldeo_lon, ldeo_lat, color='red', transform=ccrs.PlateCarree(), label="LDEO",zorder=10)
# ax.scatter(DITD_2017.ctd.bot.lon, DITD_2017.ctd.bot.lat, color='orange', transform=ccrs.PlateCarree(), label="DITD",zorder=10)
# ax.scatter(DITS_2017.ctd.bot.lon, DITS_2017.ctd.bot.lat, color='orange', transform=ccrs.PlateCarree(), label="DITS",zorder=10)
# ax.scatter(DITN_2017.ctd.bot.lon, DITN_2017.ctd.bot.lat, color='orange', transform=ccrs.PlateCarree(), label="DITN",zorder=10)
# ax.scatter(p2_lon, p2_lat, color='green', transform=ccrs.PlateCarree(), label="P2",zorder=10)
# ax.scatter(p3_lon, p3_lat, color='green', transform=ccrs.PlateCarree(), label="P3",zorder=10)

# ax.scatter(D_2017.ctd.bot.lon, D_2017.ctd.bot.lat, color='purple', transform=ccrs.PlateCarree(), label="D",zorder=10)

# try to add quiver plot for velocity 
q = ax.quiver(u_D.lon.values, u_D.lat.values, u_D, v_D, angles='xy', scale_units='xy', scale=30, color='b')
ax.quiverkey(q, X=0.9, Y=1.05, U=5, label='5cm/s', labelpos='E', coordinates='axes')

ax.quiver(u_DITD.lon.values, u_DITD.lat.values, u_DITD, v_DITD, angles='xy', scale_units='xy', scale=30, color='r')
ax.quiver(u_DITS.lon.values, u_DITS.lat.values, u_DITS, v_DITS, angles='xy', scale_units='xy', scale=30, color='g')
ax.quiver(u_DITN.lon.values, u_DITN.lat.values, u_DITN, v_DITN, angles='xy', scale_units='xy', scale=30, color='k')
ax.quiver(u_LDEO.lon.values, u_LDEO.lat.values, u_LDEO, v_LDEO, angles='xy', scale_units='xy', scale=10, color='purple')

ax.legend()

# === Inset map ===
# Use inset_axes from matplotlib
# inset_ax = inset_axes(ax, width="35%", height="35%", loc='lower left',
# #                       axes_class=plt.Axes)
# inset_ax = plt.axes([0.7, 0.235, 0.2, 0.2], projection=ccrs.SouthPolarStereo())

# inset_ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())
# inset_ax.add_feature(cfeature.LAND, facecolor='lightgrey')
# inset_ax.coastlines(resolution='110m', linewidth=0.5)

# # Rectangle showing main map extent
# main_extent = [160, 175, -76, -71]
# rect = mpatches.Rectangle((main_extent[0], main_extent[2]),
#                           main_extent[1] - main_extent[0],
#                           main_extent[3] - main_extent[2],
#                           linewidth=1.5, edgecolor='red', facecolor='none',
#                           transform=ccrs.PlateCarree(), zorder=5)
# inset_ax.add_patch(rect)


# plt.savefig('figs/TNB_Map_URSSI_small.png',format='png',dpi=400,transparent=False)


In [ ]:
plt.rcParams.update({
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 22
})

In [ ]:
DITD_2017.vel.bot.z.values

In [ ]:
fig = plt.figure(figsize=(12,12),constrained_layout=True)

plt.subplot(4,1,1)
ds = DITD_2017.vel.bot
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITD 1239 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,2)
ds = D_2017.vel.bot.sel(time=slice('2017-03','2018-03'))
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('D 1135 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,3)
ds = DITN_2017.vel.bot
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITN 675 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.subplot(4,1,4)
ds = DITS_2017.vel.bot
q = plt.quiver(
    ds.time.resample(time='W').mean().values,np.zeros(len(ds.time.resample(time='W').mean())),
    ds.u.resample(time='W').mean(),
    ds.v.resample(time='W').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITS 1088 m Velocity')
plt.ylabel('Velocity (m/s)')

plt.savefig('veloctiy_deep_sensors_2017.png')

In [ ]:
moorings['D']

In [ ]:
ds = moorings['D/2016/vel/mid'].to_dataset()

In [ ]:
ds

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ds.time,ds.v)

In [ ]:
DITD_2017.vel.bot

In [ ]:
# load in function that I previously copied from Una's code to test 


## ORIGINAL FUNCTION 

def sliding_window_correlation(ds1,ds2,lag_hours,sliding_window):

    """
    - ds1: dataset used for sliding window
    - ds2: dataset used for lag hours 

    """

    # calculate when correlation analysis should start 
    start = (ds1.time[0] + pd.Timedelta(lag_hours,'h'))
    end = (ds1.time[-1] - pd.Timedelta(lag_hours,'h'))

    # calculate sliding window indice based on lag_hours 
    
    sliding_window_indice = np.arange(-lag_hours,lag_hours+1,1)
    
    # calculate sliding window start and stop 
    window_stop = start + pd.Timedelta(sliding_window,'d')

    # calculate ndays
    ndays = (end-window_stop).values.astype('timedelta64[D]').astype('int')

    # calculate correlation stuff 
    corr = []
    pval = [] 
    
    for i in np.arange(0,ndays,1):
        start = (ds1.time[0] + pd.Timedelta(lag_hours,'h')) + pd.Timedelta(i,'d')
        stop = (ds1.time[0] + pd.Timedelta(lag_hours,'h') + pd.Timedelta(sliding_window,'d')) + pd.Timedelta(i,'d') 
        
        for hrs in range(len(sliding_window_indice)):
            
            sliding_start = ds1.sel(time=start).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
            sliding_stop = ds1.sel(time=stop).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
            
            x = signal.detrend(ds2.sel(time=slice(start,stop)))
            y = signal.detrend(ds1.sel(time=slice(sliding_start,sliding_stop)))
        
            r, p = scipy.stats.pearsonr(x, y)
            corr.append(r)
            pval.append(p)

    # convert to good format 
    corr  = np.reshape(corr, [len(np.arange(0,ndays)),len(sliding_window_indice)]) 
    pval  = np.reshape(pval, [len(np.arange(0,ndays)),len(sliding_window_indice)]) 

    start = (ds1.time[0] + pd.Timedelta(lag_hours,'h'))
    end = (ds1.time[-1] - pd.Timedelta(lag_hours,'h'))

    start_date = (ds1.time[0] + pd.Timedelta(lag_hours,'h')).values
    date_array = [start_date + np.timedelta64(i, 'D') for i in range(ndays)]


    corr = xr.DataArray(data=corr,dims=['days','lag_hours'],
             coords=dict(days=date_array,lag_hours=sliding_window_indice))

    pval = xr.DataArray(data=pval,dims=['days','lag_hours'],
                 coords=dict(days=date_array,lag_hours=sliding_window_indice))

    return corr, pval

In [ ]:
def Plot_Corr_Pval(corr,pval,title):
    
    """ Plotting function for plotting sliding window correlations"""
    
    corr_significant = corr
    corr_significant = corr_significant.where(pval < 0.05)
    fig=plt.figure(figsize=(6,6)) 
    
    cmap = cmocean.cm.balance  #CET_D1A #matplotlib.cm.Spectral_r
    bounds = np.arange(-0.9,1,.2)
    norm =  matplotlib.colors.BoundaryNorm(bounds, cmap.N)
    
    
    plt.pcolormesh(corr_significant.lag_hours,corr_significant.days,corr_significant,cmap = cmap, norm=norm )
    
    plt.gca().invert_yaxis()
    cbar = plt.colorbar()
    cbar.set_label('$R$')
    
    
    plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [0.6,1], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)
    plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [-1,-0.6], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)
    
    # plt.xlim(-100,100)
    plt.axvline(x=0,color='k',linestyle='dashed')
    
    plt.title(title)
    plt.tight_layout()



In [ ]:
ds1 = D_2017.ctd.bot.sel(time=slice('2017-04','2017-09-16')).to_dataset()

In [ ]:
ds2 = DITD_2017.ctd.bot.sel(time=slice('2017-04T00:05','2017-09-16')).to_dataset().resample(time='30min').mean(dim='time')

In [ ]:
DITD_2017.ctd.bot.sel(time=slice('2017-04T00:05','2017-09-16')).to_dataset()

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import xarray as xr 
import pandas as pd

import scipy
from scipy import signal
import colorcet as cc
import matplotlib
import cmocean 

In [ ]:
ds1.sal

In [ ]:
ds2_interp = DITD_2017.ctd.bot.sel(time=slice('2017-04T00:05','2017-09-16')).to_dataset().interp(time=ds1.time)

In [ ]:
ds2_interp

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=ds1.sal,
                                        ds2 = ds2_interp.sal,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='Mooring D and DITD Salinity',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
ds2_interp.lat

In [ ]:
dist = 21000 #dist
speed = 0.11 #m/s


In [ ]:
time = dist/speed

In [ ]:
time/3600 #convert to hours

In [ ]:
D_2017.vel.bot['Mod_0'].mean()

### Historical Context 

Questions: 

- Does the step change in salinity and temperature happen every year? Does the change in temperature always preceed the change in salinity? 

    - This can be investigated in moorings D, DITx 

- What is the magnitude of step change between the years? 

    - Look at Rusicano (or Castagno paper) for calculation on delta R (salt flux)

- Is there a link between magnitude of step change and current direction / speed? 
